# 01_ingesta_datos

## Objetivo
Este notebook realiza la ingesta inicial de los datasets del proyecto, verifica su estructura, revisa la consistencia de los nombres de los departamentos y deja preparada la base para construir el dataset maestro.

## Alcance de este notebook
- Cargar los archivos fuente desde la carpeta `data/raw/`
- Inspeccionar columnas, tipos de dato y cantidad de registros
- Estandarizar nombres de columnas clave
- Revisar la consistencia de la columna `departamento`
- Identificar diferencias entre departamentos presentes en cada fuente

## Requisitos previos
- Haber creado la estructura del proyecto
- Tener los archivos de datos dentro de `data/raw/`
- Estar usando el kernel del entorno virtual del proyecto


## 1. Importación de librerías

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Definición de rutas del proyecto



In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_RAW:', DATA_RAW)
print('DATA_PROCESSED:', DATA_PROCESSED)

## 3. Verificación de archivos esperados

En esta sección se define el nombre esperado de cada archivo fuente. 

In [ ]:
FILES = {
    'pobreza': DATA_RAW / 'pobreza_2024.xlsx',
    'microcredito': DATA_RAW / 'acceso_microcredito_2024.xlsx',
    'productos_financieros': DATA_RAW / 'acceso_productos_financieros_2024.xlsx',
    'atm': DATA_RAW / 'atm_x_10000_adultos_2024.xlsx',
    'internet': DATA_RAW / 'internet_hogares_2024.xlsx',
}

for nombre, ruta in FILES.items():
    print(f'{nombre:25s} -> {ruta.name:40s} | existe: {ruta.exists()}')

## 4. Funciones auxiliares

Estas funciones ayudan a cargar archivos, limpiar encabezados y estandarizar nombres de departamentos.

In [ ]:
def normalizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace('á', 'a', regex=False)
        .str.replace('é', 'e', regex=False)
        .str.replace('í', 'i', regex=False)
        .str.replace('ó', 'o', regex=False)
        .str.replace('ú', 'u', regex=False)
        .str.replace('ñ', 'n', regex=False)
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )
    return df


def estandarizar_departamento(serie: pd.Series) -> pd.Series:
    s = serie.astype(str).str.strip().str.upper()
    s = (s
         .str.replace('Á', 'A', regex=False)
         .str.replace('É', 'E', regex=False)
         .str.replace('Í', 'I', regex=False)
         .str.replace('Ó', 'O', regex=False)
         .str.replace('Ú', 'U', regex=False))
    reemplazos = {
        'BOGOTA D.C.': 'BOGOTA D.C.',
        'BOGOTA, D.C.': 'BOGOTA D.C.',
        'SANTAFE DE BOGOTA D.C': 'BOGOTA D.C.',
        'SANTAFE DE BOGOTA D.C.': 'BOGOTA D.C.',
        'ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA': 'SAN ANDRES Y PROVIDENCIA',
        'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA': 'SAN ANDRES Y PROVIDENCIA',
        'SAN ANDRES Y PROVIDENCIA': 'SAN ANDRES Y PROVIDENCIA',
        'NORTE DE SANTANDER': 'NORTE DE SANTANDER',
        'VALLE DEL CAUCA': 'VALLE DEL CAUCA',
        'LA GUAJIRA': 'LA GUAJIRA',
    }
    return s.replace(reemplazos)


def cargar_excel(ruta: Path, sheet_name=0) -> pd.DataFrame:
    df = pd.read_excel(ruta, sheet_name=sheet_name)
    df = normalizar_columnas(df)
    return df


def buscar_columna_departamento(df: pd.DataFrame) -> str:
    candidatas = ['departamento', 'nombre_dpt', 'dpto', 'departamentos']
    for c in candidatas:
        if c in df.columns:
            return c
    raise ValueError(f'No se encontró una columna de departamento en: {list(df.columns)}')

## 5. Carga inicial de datasets

In [ ]:
dfs = {}

for nombre, ruta in FILES.items():
    dfs[nombre] = cargar_excel(ruta)
    print('=' * 80)
    print(f'DATASET: {nombre}')
    print('Forma:', dfs[nombre].shape)
    print('Columnas:', list(dfs[nombre].columns))
    display(dfs[nombre].head())

## 6. Estandarización mínima de la columna de departamento

Aquí se crea una columna común llamada `departamento` en todos los datasets.

In [ ]:
for nombre, df in dfs.items():
    col_dep = buscar_columna_departamento(df)
    dfs[nombre] = df.copy()
    dfs[nombre]['departamento'] = estandarizar_departamento(dfs[nombre][col_dep])
    print(f'{nombre:25s} -> columna original: {col_dep}')

## 7. Revisión de departamentos por dataset

In [ ]:
departamentos_por_fuente = {}

for nombre, df in dfs.items():
    deps = sorted(df['departamento'].dropna().unique().tolist())
    departamentos_por_fuente[nombre] = set(deps)
    print('=' * 80)
    print(f'{nombre.upper()} - cantidad de departamentos únicos: {len(deps)}')
    print(deps)

## 8. Comparación entre listas de departamentos

Esta revisión permite detectar nombres inconsistentes o territorios faltantes.

In [ ]:
fuentes = list(departamentos_por_fuente.keys())
base = fuentes[0]
base_set = departamentos_por_fuente[base]

for fuente in fuentes[1:]:
    print('=' * 80)
    print(f'Comparación: {base} vs {fuente}')
    solo_base = sorted(base_set - departamentos_por_fuente[fuente])
    solo_fuente = sorted(departamentos_por_fuente[fuente] - base_set)
    print('Solo en base:', solo_base)
    print('Solo en fuente:', solo_fuente)

## 9. Selección preliminar de columnas de valor




In [ ]:
COLUMNAS_VALOR = {
    'pobreza': 'pobreza_2024',
    'microcredito': 'acceso_microcredito_2024',
    'productos_financieros': 'acceso_productos_financieros_2024',
    'atm': 'atm_x_10000_adultos_2024',
    'internet': 'internet_hogares_2024',
}

for nombre, col in COLUMNAS_VALOR.items():
    print(f'{nombre:25s} -> columna esperada: {col} | existe: {col in dfs[nombre].columns}')

## 10. Construcción de una vista preliminar estandarizada

cada archivo es reducido a:
- `departamento`
- la variable principal correspondiente

In [ ]:
vistas_limpias = {}

for nombre, df in dfs.items():
    col_valor = COLUMNAS_VALOR[nombre]
    temp = df[['departamento', col_valor]].copy()
    temp = temp.dropna(subset=['departamento'])
    temp = temp.drop_duplicates(subset=['departamento'])
    vistas_limpias[nombre] = temp
    print('=' * 80)
    print(nombre)
    print(temp.shape)
    display(temp.head())

## 11. Resumen de calidad básica

Aquí se revisan valores nulos y duplicados por dataset.

In [ ]:
resumen_calidad = []

for nombre, df in vistas_limpias.items():
    col_valor = [c for c in df.columns if c != 'departamento'][0]
    resumen_calidad.append({
        'dataset': nombre,
        'filas': len(df),
        'departamentos_unicos': df['departamento'].nunique(),
        'nulos_valor': int(df[col_valor].isna().sum()),
        'duplicados_departamento': int(df['departamento'].duplicated().sum()),
    })

resumen_calidad = pd.DataFrame(resumen_calidad)
display(resumen_calidad)